# Mini Project 7 — UK Online Retail: A Take-Home Style Pipeline (Bronze → Silver → Gold)

A **real dataset** this time (~540,000 actual transactions from a UK online retailer),
and a different format: this project is structured like a **data engineering take-home
assignment** — no task list, only business requirements. I had to profile the data,
decide the steps myself, and defend every decision in a written decision log.

**Techniques in this project:**
- Requirements-driven pipeline design (no step-by-step instructions)
- Reading everything as **string** (no `inferSchema`) to keep full control over types
- Profiling before cleaning: grain proof, null counts, numeric ranges
- **Business-rule filtering**: returns/cancellations and adjustments are real events, but not sales
- Keeping 25% of rows with a null `CustomerID` — and being able to say why
- Full-row `dropDuplicates()` matched to the table grain (lesson carried over from project 6)
- `decimal(10,2)` for money, verified single-format timestamp parsing
- Gold layer: monthly revenue per country + top-10 products, in **SQL and DataFrame API**
- Idempotent `overwrite` writes (rerunning the notebook must not double the numbers)
- A **decision log** (R5) written for a code-review style defense

## The assignment

**Company:** NorthCart Ltd. (fictional UK online retailer) · **Role:** Data Engineer · **Platform:** Databricks

> *"Build us a pipeline we can trust. Take the raw export, produce a clean table for
> the analysts, and give finance the summary numbers they keep asking for. Document
> every decision you make — I will review your work like a code review."*

Every transaction is exported from a legacy order system as a CSV. Nobody has ever
checked the quality of this export. The analytics team says the numbers "look wrong"
but they can't say why.

**Dataset (real):** [Kaggle — E-Commerce Data (UK online retail)](https://www.kaggle.com/datasets/carrie1/ecommerce-data), ~541k rows.

### Business requirements

- **R1 — Raw layer:** keep an untouched copy of the data exactly as it arrived.
- **R2 — Clean sales table:** correct types, consistent text, every row a **genuine
  completed sale**, plus the **total value of each invoice line**.
- **R3 — Finance summary:** monthly revenue per country + top 10 products by revenue.
- **R4 — Re-runnable:** running the pipeline twice must not double the results.
- **R5 — Decision log:** for every row removed or value changed — what, how, **why**.

### Rules I worked under

1. **No task list.** Read the requirements, profile the data, decide the steps yourself.
2. **Profile before you clean.** Count and measure problems before deleting anything.
3. **Deleting data is a business decision, not a technical one.** Investigate before you drop.

### Data dictionary (all the documentation that exists)

| Column | What the legacy team says it is |
|---|---|
| `InvoiceNo` | Invoice number of the transaction |
| `StockCode` | Product code |
| `Description` | Product name |
| `Quantity` | Units sold in that line |
| `InvoiceDate` | When the invoice was generated |
| `UnitPrice` | Price per unit (GBP) |
| `CustomerID` | Customer number |
| `Country` | Customer country |

## 1 · Load the raw data

Everything is read as **string** on purpose (no `inferSchema`). With an untrusted
legacy export I want to *see* what the values look like before Spark guesses types —
a silent wrong guess (e.g. `CustomerID` as double: `17850.0`) is worse than casting
manually later.

In [ ]:
retail_raw = (
    spark.read
    .option("header", True)
    .format("csv")
    .load("/Volumes/dev/spark_db/datasets/mini-projects/raw_data/uk_online_retail_raw.csv")
)
retail_raw.display()

## 2 · Profiling — numbers first, decisions second

Before touching anything: schema, row count, grain, nulls, value ranges.
Every cleaning decision below is backed by a number measured here.

In [ ]:
retail_raw.printSchema()   # every column is string — as expected with no inferSchema

In [ ]:
print(retail_raw.count())        # 541,909 rows
print(len(retail_raw.columns))   # 8 columns

### Grain proof

What does one row represent? The row count vs distinct counts tell the story:

- `InvoiceNo + CustomerID` is **not** unique → one invoice has many rows (~21 lines per invoice)
- `StockCode` distinct = 4,070 → not one row per product either
- Full-row distinct < row count → there are **exact duplicate rows**

Conclusion: **one row = one invoice line** (one product sold on one invoice),
and the export contains duplicated lines that need investigating.

In [ ]:
print(retail_raw.count())                                          # 541,909 total rows
print(retail_raw.select("InvoiceNo", "CustomerID").distinct().count())  # 25,900 invoices -> ~21 lines each
print(retail_raw.select("StockCode").distinct().count())           # 4,070 products
print(retail_raw.distinct().count())                               # full-row distinct -> duplicates exist

### Null check

`CustomerID` has ~135k nulls (~25% of the table) and `Description` has 1,454.
Every other column is fully populated. Whether these nulls are a *problem* depends
on the goal — see the decision log.

In [ ]:
from pyspark.sql.functions import col

for c in retail_raw.columns:
    print(c, retail_raw.filter(col(c).isNull()).count())

In [ ]:
# What do the null-customer rows look like? Product, quantity, price, date are all
# present -> these are real sales (likely guest/cash orders), just with no customer id.
retail_raw.filter(col("CustomerID").isNull()).display()

### Problems found

1. Column names are legacy `CamelCase` → rename to `snake_case`
2. All column types are string → cast to proper types
3. Exact duplicate rows exist → grain problem, investigate and dedupe
4. Nulls in `CustomerID` (~25%) and `Description` — **not** a problem for revenue
   calculations (decision log #1 and #2)
5. Negative values in `Quantity` and `UnitPrice` (found below) → not every row is a sale

## 3 · R1 — Raw layer

An untouched copy, written **before** any rename or cast. If someone questions the
numbers later, this is the original I can point to.

In [ ]:
retail_raw.write.mode("overwrite").saveAsTable("dev.mini_projects.uk_retail_raw")

In [ ]:
spark.read.table("dev.mini_projects.uk_retail_raw").display()

## 4 · R2 — Clean sales table

Cleaning plan (in this order):
1. Column names → `snake_case`
2. Normalize text values (`trim`, case rules)
3. Cast types (clean first, cast second)
4. Business-rule filter: keep only genuine completed sales
5. Deduplicate at the correct grain
6. Add `total_value` and write the Silver table

In [ ]:
retail_renamed = retail_raw.withColumnsRenamed({
    "InvoiceNo"    : "invoice_no",
    "StockCode"    : "stock_code",
    "Description"  : "description",
    "Quantity"     : "quantity",
    "InvoiceDate"  : "invoice_date",
    "UnitPrice"    : "unit_price",
    "CustomerID"   : "customer_id",
    "Country"      : "country"
})
retail_renamed.display()

### Normalize text

Rules: `trim` everything; `lower` only free-text (`description`); `initcap` for
`country` so `UNITED KINGDOM` / `united kingdom` group as one value.

**Identifier columns (`invoice_no`, `stock_code`) are NOT lowercased** — case can be
meaningful in codes. In this dataset cancelled invoices are prefixed with `C`
(e.g. `C536379`), and stock codes mix letters and digits. Changing their case would
rewrite identifiers that other systems may join on.

In [ ]:
from pyspark.sql.functions import col, trim, lower, initcap

normalized_retail_raw = retail_renamed.withColumns({
    "invoice_no"    : trim(col("invoice_no")),
    "stock_code"    : trim(col("stock_code")),
    "description"   : trim(lower(col("description"))),
    "quantity"      : trim(col("quantity")),
    "invoice_date"  : trim(col("invoice_date")),
    "unit_price"    : trim(col("unit_price")),
    "customer_id"   : trim(col("customer_id")),
    "country"       : trim(initcap(col("country")))
})
normalized_retail_raw.display()

### Cast types — clean first, cast second

- `quantity` → `int`
- `unit_price` → `decimal(10,2)` — money needs exact values, never float
- `invoice_date` → `timestamp` with format `M/d/yyyy H:mm` (single-letter tokens so
  both 1- and 2-digit month/day/hour parse: `12/1/2010 8:26` and `1/4/2011 14:32`)

I verified the format first with `try_to_timestamp`: **0 unparseable rows**, so one
format is enough — no need for a multi-format `coalesce` fallback here. (A fallback
format you haven't proven you need is a risk, not a safety net: an ambiguous second
pattern like `d/M/yyyy` could silently mis-parse dates instead of failing loudly.)

In [ ]:
from pyspark.sql.functions import col, to_timestamp

normalized_retail_raw = normalized_retail_raw.withColumns({
    "quantity"     : col("quantity").cast("int"),
    "invoice_date" : to_timestamp(col("invoice_date"), "M/d/yyyy H:mm"),
    "unit_price"   : col("unit_price").cast("decimal(10,2)")
})
normalized_retail_raw.display()

### Numeric ranges — are all rows really sales?

`Quantity` goes down to **−80,995** and `UnitPrice` down to **−11,062.06**.
Negative quantity = returns/cancellations; negative or zero price = adjustments,
bad debt postings, free items. So no — not every row is a sale.

In [ ]:
from pyspark.sql.functions import max, min

normalized_retail_raw.select(max("quantity"), min("quantity")).display()
normalized_retail_raw.select(max("unit_price"), min("unit_price")).display()

### Business-rule filter: keep only genuine completed sales

R2 says every row must be a **completed sale**. A return is a real business event,
but it is not a sale — it does not belong in a sales table. Same for zero/negative
price adjustments.

Decision: **drop** these rows (541,909 → 530,100, −11,809). I did *not* null the
values — nulling would leave non-sale rows sitting in a sales table. (Different
call than project 6, where a broken *value* on a real event meant: keep the row,
null the value. Here the whole *row* fails the business definition.)

In [ ]:
normalized_retail_raw = normalized_retail_raw.filter((col("unit_price") > 0) & (col("quantity") > 0))
normalized_retail_raw.count()

### Deduplicate at the correct grain

The grain is one invoice line, and there is no line-number column — so two identical
rows are indistinguishable. 5,226 rows are exact duplicates across all 8 columns:
in a legacy export that is almost always a double-export artifact, and counting a
sale twice inflates finance numbers.

**Full-row `dropDuplicates()`** — no column subset. (Project 6 lesson: subset-dedup
on the wrong grain silently deletes real events.)

In [ ]:
print(normalized_retail_raw.count())             # 530,100 before dedup
print(normalized_retail_raw.distinct().count())  # 524,874 distinct -> 5,226 exact duplicates

In [ ]:
silver_retail_df = normalized_retail_raw.dropDuplicates()
silver_retail_df.count()

### Add the line total and write Silver

`total_value = quantity * unit_price` — the "value of each invoice line" the
analysts asked for. A per-row multiplication, so the row count stays 524,874.

Written with `overwrite` (+ `overwriteSchema` once, because adding `total_value`
changed the schema) → rerunning the notebook rebuilds the same table (R4).

In [ ]:
silver_retail_df.createOrReplaceTempView("silver_view")

silver_final_df = spark.sql("""
    SELECT invoice_no, stock_code, description, invoice_date, customer_id, country,
           unit_price, quantity,
           (quantity * unit_price) AS total_value
    FROM silver_view
""")

silver_final_df.write.mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable("dev.mini_projects.uk_retail_silver")

In [ ]:
print(silver_retail_df.count())   # 524,874
print(silver_final_df.count())    # 524,874 -> adding a column changed no row counts

## 5 · R3 — Gold layer: finance summaries

Both reports are written twice — Spark SQL and DataFrame API — on purpose, to keep
both muscles fresh. `year_month` uses `yyyy-MM` so the string sorts chronologically.

### Monthly revenue per country — SQL

In [ ]:
%sql
SELECT country,
       date_format(invoice_date, 'yyyy-MM')     AS year_month,
       round(sum(total_value), 2)               AS total_revenue
FROM dev.mini_projects.uk_retail_silver
GROUP BY country, date_format(invoice_date, 'yyyy-MM')
ORDER BY country, year_month

### Monthly revenue per country — DataFrame API

In [ ]:
from pyspark.sql.functions import col, date_format, sum, round

monthly_revenue_per_country = (
    silver_final_df
    .withColumn("year_month", date_format("invoice_date", "yyyy-MM"))
    .groupBy("country", "year_month")
    .agg(round(sum(col("total_value")), 2).alias("total_revenue"))
)
monthly_revenue_per_country.display()

In [ ]:
monthly_revenue_per_country.write.mode("overwrite") \
    .saveAsTable("dev.mini_projects.monthly_revenue_per_country")

### Top 10 products by total revenue — SQL

In [ ]:
%sql
SELECT stock_code AS product,
       sum(total_value) AS total_revenue
FROM dev.mini_projects.uk_retail_silver
GROUP BY stock_code
ORDER BY total_revenue DESC
LIMIT 10

### Top 10 products by total revenue — DataFrame API

In [ ]:
total_revenue_top_ten_df = (
    silver_final_df
    .groupBy("stock_code")
    .agg(sum(col("total_value")).alias("total_revenue"))
    .orderBy("total_revenue", ascending=False)
    .limit(10)
)
total_revenue_top_ten_df.display()

In [ ]:
total_revenue_top_ten_df.write.mode("overwrite") \
    .saveAsTable("dev.mini_projects.total_revenue_top_ten")

### Observation on the top-10 result

Some of the top `stock_code` values — `DOT`, `POST`, `M` — are **service/adjustment
codes** (postage, manual entries), not real products. In a stricter version of the
"top products" report I would exclude non-product stock codes. **Flagged, not
removed** — that exclusion is a reporting decision the product team should confirm.

# R5 — Decision Log

Pipeline: **bronze (raw) → silver (clean sales) → gold (finance summaries)**
Raw file: `online_retail.csv` (541,909 rows). Everything read as **string** (no `inferSchema`) to keep full control over types and avoid silent wrong guesses.

## Profiling (before any cleaning)
- **Grain:** one row = one invoice line (one product sold on one invoice). Proven: `InvoiceNo + CustomerID` is not unique (25,900 distinct vs 541,909 rows → ~21 lines per invoice). Distinct `StockCode` = 4,070 (not one row per product).
- **Nulls:** `CustomerID` = 135,080 (~25%), `Description` = 1,454. All other columns 0 nulls.
- **Numeric ranges (after cast):** `Quantity` min −80,995 / max 80,995. `UnitPrice` min −11,062.06 / max 38,970. Negatives present → not all rows are sales.

## Decisions

**1. Null CustomerID (135,080 rows) — KEPT.**
Found: ~25% of rows have no customer id. What I did: kept them. Why: these are still real sales (product, quantity, price, date all present) — likely guest/cash/wholesale orders. My goal (revenue by country, top products) does not need the customer id. Dropping 25% of real sales would understate revenue. I did not impute a fake id — null honestly means "unknown customer".

**2. Null Description (1,454 rows) — KEPT.**
Why: the top-products report groups by `StockCode` (the stable identifier), not by description, and revenue does not depend on the product name. So a missing name does not make the row invalid.

**3. Data types — cast from string.**
`Quantity` → int, `UnitPrice` → decimal(10,2) (money needs exact values, not float), `InvoiceDate` → timestamp. Text columns trimmed; `Country` normalised with `initcap`. Identifier columns (`StockCode`, `InvoiceNo`) NOT lowercased — case can be meaningful in codes (cancelled invoices carry a `C` prefix).

**4. Date parsing.**
Format: `M/d/yyyy H:mm` (single-letter tokens so both 1- and 2-digit month/day/hour parse; data mixes e.g. `12/1/2010 8:26` and `1/4/2011 14:32`). Verified with `try_to_timestamp` → 0 unparseable rows, so one format is enough (no need for a multi-format coalesce, and an unproven fallback format could silently mis-parse).

**5. Non-sales removed (11,809 rows) — business rule filter.**
Found: rows with `quantity <= 0` (returns/cancellations) or `unit_price <= 0` (adjustments / bad debt / free items). What I did: kept only `quantity > 0 AND unit_price > 0`. Rows: 541,909 → 530,100 (−11,809). Why: R2 requires a **completed sale** on every row. A return is a real business event but it is not a sale, so it does not belong in a sales table. I removed these rows (I did NOT null the values — nulling would leave non-sale rows in the table). Note: returns were fully excluded, not netted against sales — that is a separate design choice.

**6. Exact duplicates removed (5,226 rows).**
Found: 5,226 rows identical across all 8 columns. What I did: `dropDuplicates()` (one copy kept per group). Rows: 530,100 → 524,874. Why: identical rows in a legacy export are almost always double-export artifacts; counting a sale twice would inflate finance numbers.

**7. Line total added.**
New column `total_value = quantity * unit_price` (decimal). Per-row value (multiplication), so row count is unchanged (524,874). This is the "value of each invoice line" the analysts asked for.

## Gold layer (R3)
- **Monthly revenue per country:** `groupBy(country, year_month)` + `sum(total_value)` (302 rows). `year_month` derived with `date_format(..., "yyyy-MM")` so the string sorts chronologically.
- **Top 10 products by revenue:** `groupBy(stock_code)` + `sum(total_value)`, ordered descending, limit 10.

## Open observations (for review)
- Top-10 products include `DOT`, `POST`, `M` — these look like service/adjustment codes (postage, manual entries), **not real products**. In a stricter analysis I would exclude non-product stock codes from the "top products" report. Flagged, not removed.

## Idempotency (R4)
All tables written with `mode("overwrite")` → re-running the notebook rebuilds the same tables, no duplicated data. Silver needed `option("overwriteSchema", "true")` once because the schema changed (added `total_value`).

## Key takeaways

1. **A take-home has no task list.** Requirements → profile → plan → execute.
   Structuring the work is part of the work.
2. **Profile before you clean.** Every decision above is backed by a number
   measured *before* anything was deleted.
3. **Deleting data is a business decision.** Returns are real events but not
   sales — dropped with counts documented (−11,809), not silently filtered.
4. **A null is not automatically a problem.** 25% of rows have no `CustomerID`
   and they all stayed: dropping them would understate revenue by a quarter.
5. **Read as string, cast deliberately.** And *verify* the date format
   (`try_to_timestamp` → 0 failures) before trusting it — don't add fallback
   formats you haven't proven you need.
6. **Dedup at the grain.** Full-row `dropDuplicates()` removed 5,226 double-export
   artifacts without risking real invoice lines.
7. **Idempotent writes** (`overwrite`) mean the manager can rerun the notebook
   and the numbers don't double.
8. **The decision log is the deliverable.** A pipeline you can't defend line by
   line in review is a pipeline nobody will trust.